In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import sys
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
sys.path.insert(0, 'src')
from entropy_metrics import calc_shannon
from drift_detection import calc_delta_h
from risk_signal import generate_risk_score
from iot_producer import IoTProducer
from digital_twin import DigitalTwinConsumer

# Load the baseline saved from Notebook 1
with open('results/logs/baseline_metrics.json', 'r') as f:
    baseline = json.load(f)
print(f"baseline {baseline}")

# Configuration
WINDOW_SIZE = 500
STEP = 100
BROKER = "kafka:9092"
TOPIC = "drift_telemetry"

In [ ]:
# Load drift dataset and send to Kafka
df = pd.read_csv('data/drift_dataset.csv')
signal = df['sensor_value'].values
print(f"Dataset di drift caricato: {len(signal)} campioni")

# Initialize producer and send drift data to Kafka
import time
producer = IoTProducer(broker=BROKER, topic=TOPIC)
print(f"📤 Invio {len(signal)} campioni di drift a Kafka...")
start = time.time()
for i, value in enumerate(signal):
    producer.send_data(timestamp=i, value=value)
    if (i + 1) % 1000 == 0:
        print(f"  → {i + 1}/{len(signal)} inviati")
producer.flush()
elapsed = time.time() - start
print(f"✅ {len(signal)} campioni inviati in {elapsed:.2f}s")

# Consume data from Kafka and calculate entropy and risk
consumer = DigitalTwinConsumer(broker=BROKER, topic=TOPIC, group="entropy-analysis-v2")
consumer.set_baseline(baseline)
history = []

# Collect metrics directly from digital twin
max_collect_attempts = 100
attempts = 0
while attempts < max_collect_attempts:
    metrics = consumer.analyse_metrics(window_size=WINDOW_SIZE, timeout_ms=5000)
    if metrics is not None:
        history.append(metrics)
        #print(f"  → Analizzata finestra {len(entropy_history)}: Shannon={metrics['shannon']:.4f}")
    else:
        if len(history) > 0:
            break
    attempts += 1

print(f"Analisi completata: {len(history)} finestre elaborate")

consumer.close()

h_t = np.array([h['shannon'] for h in history])
delta_h_list = np.array([h['delta_h'] for h in history])
risk_levels = np.array([h['risk_level'] for h in history])

print(f"\n✅ Analisi completata. Rilevati {len(risk_levels)} segmenti.")

In [ ]:
if len(h_t) == 0:
    print("⚠️  Nessun dato da visualizzare")
else:
    plt.figure(figsize=(12, 8))

    # Plot 1: Entropia vs Baseline
    plt.subplot(2, 1, 1)
    plt.plot(h_t, color='blue', label='H(t) corrente')
    plt.axhline(baseline['shannon_mean'], color='green', linestyle='--', label='Baseline')
    plt.title("Rilevazione Drift tramite Variazione Entropica")
    plt.legend()

    # Plot 2: Segnale di Rischio
    plt.subplot(2, 1, 2)
    plt.plot(delta_h_list, color='orange', label='ΔH(t)')
    plt.axhline(baseline['shannon_std'] * 2, color='orange', linestyle=':', label='Soglia Warning (2σ)')
    plt.axhline(baseline['shannon_std'] * 3, color='red', linestyle=':', label='Soglia Critical (3σ)')
    plt.fill_between(range(len(risk_levels)), 0, np.array(delta_h_list), 
                     where=np.array(risk_levels)==2, color='red', alpha=0.3)
    plt.title("Generazione Segnale di Rischio")
    plt.xlabel("Window Index")
    plt.legend()

    plt.tight_layout()
    plt.savefig('results/figures/drift_analysis.png')
    print("✅ Grafico salvato in results/figures/drift_analysis.png")
    plt.show()